<a href="https://colab.research.google.com/github/takatakamanbou/ML/blob/2025/AdvML2025_ex13notebookA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AdvML ex13notebookA

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/AdvML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?AdvML)




板書や口頭で補足する前提なので，この notebook だけでは説明が不完全です．


---
## 準備
---


今回の notebook で行う実験の中には，それなりに実行時間が長くなるものもある．GPU を利用した方が短い時間で済ませられるので，次のようにランタイムのタイプを変更しよう．

1. Colab のメニューから「ランタイム」>「ランタイムのタイプを変更」>「CPU」に代えて「T4 GPU」を選択して「保存」
1. ランタイムが初期化されるので，必要なら全てのコードセルを最初から実行し直す


In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

# scikit-learn のいろいろ
from sklearn.datasets import make_moons, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture

# NumPy の 疑似乱数生成器（rng = random number generator）
from numpy.random import default_rng
rng = default_rng() # 疑似乱数生成器を初期化

# PyTorch 関係のほげ
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
import torchsummary

次のコードセルを実行して，`cuda` と表示されれば， PyTorch で GPU が使える状態になっている．`cpu` と表示される場合は，GPU が使えるようになっておらず，PyTorch のプログラムは CPU で実行される．


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

---
## 変分オートエンコーダ
---

あるデータ $\pmb{x}$ の確率密度 $p(\pmb{x})$ を，$\pmb{\theta}$ というパラメータを持つ確率モデル $p_{\pmb{\theta}}(\pmb{x})$ によって推定したとする．このとき，モデルの性質によっては，$p_{\pmb{\theta}}(\pmb{x})$ に従う $\pmb{x}$ を無限に生成することができる．このように，観測されたデータの分布を学習し，その分布に従って新たなデータを生成できるようなモデルを，**生成モデル**（generative model）と呼ぶ．

すでに説明したように，正規分布やGMMは単純な生成モデルの例であり，これらでは明示的に確率密度関数 $p(\pmb{x})$ が定義されている．一方で，深層学習を用いると，より柔軟な生成モデルを作ることができる．ここでは，そのような生成モデルの一例として，**変分オートエンコーダ** （**Variational Auto-Encoder, VAE**）を扱う（注）．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注: この節の内容は，次の書籍に基づいています．より詳しく知りたいひとはそちらを参考にしてください：「ゼロから作る Deep Learning (5) — 生成モデル編」 斎藤康毅，オライリージャパン，2024.
</span>

---
### 実験: GMM による手書き数字画像の生成

VAE の話を始める前に，GMM を用いて手書き数字画像を生成する実験を行ってみよう．

In [ ]:
# MNIST データセットの入手
Xraw, yraw = fetch_openml('mnist_784', version=1, parser='auto', return_X_y=True, as_frame=False)
Xall = Xraw[:20000] / 255.0     # 画素値が [0, 255] の整数値なので [0, 1] の浮動小数点数値に変換
yall = yraw[:20000].astype(int) # クラスラベル．0 から 9 の整数値

# 学習データとテストデータの分割
XL, XT, yL, yT = train_test_split(Xall, yall, test_size=4000, random_state=4649, stratify=yall)
print(XL.shape, yL.shape)
print(XT.shape, yT.shape)
NL, D = XL.shape
NT = len(XT)

#K = 10

# 平均を引いたデータを用意
Xm = np.mean(XL, axis=0)
XL2 = XL - Xm
XT2 = XT - Xm

In [ ]:
# 学習データの最初の50枚を可視化
nrow, ncol = 5, 10
fig, ax = plt.subplots(nrow, ncol, figsize=(0.6*ncol, 0.6*nrow))
for i in range(nrow):
    for j in range(ncol):
        img = XL[i*ncol + j, ::].reshape((28, 28))
        ax[i, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
        ax[i, j].axis('off')

fig.tight_layout()
plt.show()

次のコードセルを実行すると，上記のデータに GMM を当てはめる．

In [ ]:
##### GMM のパラメータの推定
M = 50
covtype = 'diag'
gmm = GaussianMixture(n_components=M, covariance_type=covtype, verbose=2, verbose_interval=10)
gmm.fit(XL)

In [ ]:
##### GMM による生成

nrow, ncol = 5, 10
N = nrow * ncol
D = XL.shape[1]

# データを生成
Z = rng.choice(M, size=N, p=gmm.weights_)
XXrec = np.empty((N, D))
for m in range(M):
    Nm = np.sum(Z == m)
    mu = gmm.means_[m]
    if covtype == 'full':
        cov = gmm.covariances_[m]
    elif covtype == 'diag':
        cov = np.diag(gmm.covariances_[m])
    elif covtype == 'spherical':
        cov = gmm.covariances_[m] * np.eye(D)
    XXrec[Z == m, :] = rng.multivariate_normal(mu, cov, size=Nm)

# 生成した画像を可視化
fig, ax = plt.subplots(nrow, ncol, figsize=(0.6*ncol, 0.6*nrow))
for i in range(nrow):
    for j in range(ncol):
        img = XXrec[i*ncol + j, ::].reshape((28, 28))
        ax[i, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
        ax[i, j].axis('off')

fig.tight_layout()
plt.show()

---
### VAE とは

ニューラルネットを用いる生成モデルの一種．連続な潜在変数 $\pmb{z} \in \mathbb{R}^{H}$ が非線形変換を受けて観測変数 $\pmb{x} \in \mathbb{R}^{D}$ となると考える．$\pmb{z}$ から $\pmb{x}$ への変換をニューラルネットでモデル化する．このニューラルネットは AE におけるデコーダに相当するので，VAE でもデコーダと呼ぶ．

VAE では，潜在変数 $\pmb{z}$ は，ある決まった（パラメータ固定の）正規分布に従うと仮定する．
具体的には，平均 $\pmb{0}$，分散共分散行列が単位行列 $I$ の正規分布に従うものとする．
つまり，

$$
p(\pmb{z}) = N(\pmb{z}; \pmb{0}, I)
$$

この正規分布から得られたある値 $\pmb{z}$ をデコーダに入力して得られる出力を

$$
\hat{\pmb{x}} = \textrm{Dec}(\pmb{z}; \pmb{\theta})
$$

と表すことにする．$\hat{\pmb{x}} \in \mathbb{R}^{D}$ である．また，$\pmb{\theta}$ は，デコーダのパラメータを表す．

このとき，観測変数 $\pmb{x}$ は，平均が $\hat{\pmb{x}}$ で分散共分散行列が $I$ の正規分布に従うと仮定する．つまり，

$$
p(\pmb{x}) = N(\pmb{x}; \hat{\pmb{x}}, I)
$$

これが，VAEによるデータ生成のモデルである．VAE では，個々のデータは次の過程によって生成される．

1. 平均 $0$ 分散共分散行列 $I$ の正規分布からひとつの値 $\pmb{z}$ を得る
1. それがデコーダによって変換されて $\hat{\pmb{x}}$ になる
1. 平均 $\hat{\pmb{x}}$ 分散共分散行列 $I$ の正規分布からひとつの値 $\pmb{x}$ を得る



---
### VAE の学習

VAE の学習は，正規分布やGMMと同様に，対数尤度の最大化によって行う．

学習データを $\{ \pmb{x}_n \}_{n = 1}^N$ とすると，VAE モデルの対数尤度
は次のような式となる．

$$
\sum_{n=1}^{N} \log p_{\pmb{\theta}}(\pmb{x}_n) = \sum_{n=1}^{N} \log \int p_{\pmb{\theta}} (\pmb{x}_n, \pmb{z}) d\pmb{z} = \sum_{n=1}^{N} \log \int p_{\pmb{\theta}} (\pmb{x}_n|\pmb{z})p(\pmb{z}) d\pmb{z}
$$

第2項，第3項には連続な変数 $\pmb{z}$ の積分が含まれているので，これらの式に基づいてパラメータの最適化を行うのは難しい．

そこで，EMアルゴリズムで考えたのと同様に，任意の確率分布 $q(\pmb{z})$ を用いて式を変形する．以下，ひとつのデータに対する対数尤度 $\log p_{\pmb{\theta}}(\pmb{x})$ について考える（添え字 $n$ も省略）．

$$
\begin{aligned}
\log p_{\pmb{\theta}}(\pmb{x}) &=  \int q(\pmb{z})\log \frac{p_{\pmb{\theta}}(\pmb{x}, \pmb{z})}{q(\pmb{z})} d\pmb{z} + \int q(\pmb{z}) \log \frac{q(\pmb{z})}{p_{\pmb{\theta}}(\pmb{z}|\pmb{x})} d\pmb{z}\\
&= \textrm{ELBO} + D_{\textrm KL}(q(\pmb{z})\Vert p_{\pmb{\theta}}(\pmb{z}|\pmb{x}))
\end{aligned}
$$

式変形の過程は省略した（EMアルゴリズムの解説にある類似の式変形の $\sum$ を $\int$ に置き換えたものになっている）．

このように ELBO + KL-divergence の形になるので，EMアルゴリズムと同じ手続きで最適化を行うことも考えられるが，まだ困難がある（詳細は省略）．そこで，VAEでは，$q(\pmb{z})$ を正規分布に限定し，その限定のもとで ELBO を最大化する，というアプローチをとる．
それでもまだ， $q(\pmb{z})$ のパラメータ（平均と分散共分散行列）をデータごとに用意しないといけないという困難があるので，ニューラルネットを用いて，これらのパラメータをデータ $\pmb{x}$ から求めることにする．

この役割を担うニューラルネットをエンコーダと呼ぶ．エンコーダは，$\pmb{x}$ を入力すると，$\pmb{x}$ に対応する $q(\pmb{z})$ の平均と分散共分散行列を出力する．ただし，簡単のため， $q(\pmb{z})$ の分散共分散行列は対角行列に限定する．このエンコーダは次式で表される．

$$
\pmb{\mu}, \pmb{\sigma} = \textrm{Enc}(\pmb{x}; \pmb{\phi})
$$

$\pmb{\mu}$ は，$\pmb{x}$ に対応する $q(\pmb{z})$ の平均であり，$\pmb{\sigma}$ は，$q(\pmb{z})$ の分散共分散行列の対角要素の平方根（すなわち$\pmb{z}$ の各要素の標準偏差）をならべたベクトルである（$\pmb{\sigma} = (\sigma_1, \sigma_2, \ldots, \sigma_H)$）．また，$\pmb{\phi}$ は，エンコーダのパラメータを表す．
$\pmb{x}$ が与えられたときの $q(\pmb{z})$ は，次式のように表される．

$$
q_{\pmb{\phi}}(\pmb{z}|\pmb{x}) = N(\pmb{z}; \pmb{\mu}, \pmb{\sigma}^2I)
$$

ここで，$\pmb{\sigma}^2 I$ という式は，ベクトル $\sigma$ の要素ごとの2乗を対角要素にもつ対角行列を表している（が，数学的に正しい表現ではないので注意）．

VAEでは，以上のように定式化した上で，ELBO の最大化を行う．
$q$ はエンコーダでモデル化されたので，ELBO のパラメータは，デコーダのパラメータ $\pmb{\theta}$ とエンコーダのパラメータ $\pmb{\phi}$ である．あるデータ $\pmb{x}$ のELBO の値を $\textrm{ELBO}(\pmb{x}; \pmb{\theta}, \pmb{\phi})$ と表すと，これは次のように変形できる．


$$
\begin{aligned}
\textrm{ELBO}(\pmb{x}; \pmb{\theta}, \pmb{\phi}) &=  \int q_{\pmb{\phi}}(\pmb{z}|\pmb{x})\log \frac{p_{\pmb{\theta}}(\pmb{x}, \pmb{z})}{q_{\pmb{\phi}}(\pmb{z}|\pmb{x})} d\pmb{z} =
\int q_{\pmb{\phi}}(\pmb{z}|\pmb{x})\log \frac{p_{\pmb{\theta}}(\pmb{x}|\pmb{z})p(\pmb{z})}{q_{\pmb{\phi}}(\pmb{z}|\pmb{x})} d\pmb{z}\\
&= \int q_{\pmb{\phi}}(\pmb{z}|\pmb{x})\log p_{\pmb{\theta}}(\pmb{x}|\pmb{z})d\pmb{z} - \int q_{\pmb{\phi}}(\pmb{z}|\pmb{x})\log \frac{q_{\pmb{\phi}}(\pmb{z}|\pmb{x})}{p(\pmb{z})} d\pmb{z}\\
&= \underbrace{\textrm{E}_{q_{\pmb{\phi}}(\pmb{z}|\pmb{x})}[ \log p_{\pmb{\theta}}(\pmb{x}|\pmb{z}) ]}_{J_1} - \underbrace{D_{\textrm KL}(q_{\pmb{\phi}}(\pmb{z}|\pmb{x}) \Vert p(\pmb{z}) )}_{J_2}\\
\end{aligned}
$$

上に示すように，最下行の2つの項を $J_1, J_2$ と表記する．



$J_1$ は，分布 $q_{\pmb{\phi}}(\pmb{z}|\pmb{x})$ のもとでの $\log p_{\pmb{\theta}}(\pmb{x}|\pmb{z}) $ の期待値である．
コンピュータを用いてこのような期待値を求める場合，「モンテカルロ法」を利用して近似値を求めることがよく行われる．$J_1$ の近似値をモンテカロル法によって求める場合，分布 $q_{\pmb{\phi}}(\pmb{z}|\pmb{x})$ に従う乱数でいくつか $\pmb{z}$ のサンプルを生成し，それらを用いて $\log p_{\pmb{\theta}}(\pmb{x}|\pmb{z})$ の平均を求めることになる．
詳しく書くと次の手続きとなる：

1. $\pmb{x}$ をエンコーダへ入力して， $\pmb{\mu}$ と $\pmb{\sigma}$ を得る．
1. $N(\pmb{z}; \pmb{\mu}, \pmb{\sigma}^2I)$ に従う $S$ 個のサンプル $\pmb{z}_1, \pmb{z}_2, \ldots, \pmb{z}_S$ を生成する．
1. それらをデコーダへ入力して，$\hat{\pmb{x}}_1, \hat{\pmb{x}}_2, \ldots, \hat{\pmb{x}}_S$ を得る．
1. $J_1 \approx \frac{1}{S} \sum_{s=1}^{S}\log N(\pmb{x}|\hat{\pmb{x}}_s, I)$ とする．

ここでは $S$ 個のサンプルを用いるとしているが，VAEにおいては，実用上 $S=1$ で十分なことも多い．
その場合，

$$
\begin{aligned}
J_1 & \approx \log N(\pmb{x}|\hat{\pmb{x}}_s, I) \\
&= \log \left( \frac{1}{\sqrt{(2\pi)^D |I|}} \exp \left(-\frac{1}{2}(\pmb{x} - \hat{\pmb{x}})^{\top} I^{-1} (\pmb{x} - \hat{\pmb{x}})\right) \right) \\
&= -\frac{1}{2}(\pmb{x} - \hat{\pmb{x}})^{\top} (\pmb{x} - \hat{\pmb{x}}) - \log \sqrt{(2\pi)^D} \\
&= -\frac{1}{2}\Vert \pmb{x} - \hat{\pmb{x}} \Vert^2 - \frac{D}{2}\log{2\pi}
\end{aligned}
$$

となる．最下行の第2項は定数である．したがって，あるデータ $\pmb{x}$ に対する $J_1$ の値は，そのデータと，それをエンコーダ→デコーダに入力して得られる再構成 $\hat{\pmb{x}}$ との間の二乗誤差が小さくなればなるほど大きくなる．

一方，$J_2$ は，$q_{\pmb{\phi}}(\pmb{z}|\pmb{x})$ の $p(\pmb{z})$ に対する KL-divergence である．
2つの分布は


$$
\begin{aligned}
q_{\pmb{\phi}}(\pmb{z}|\pmb{x}) &= N(\pmb{z}; \pmb{\mu}, \pmb{\sigma}^2I) \\
p(\pmb{z}) &= N(\pmb{z}; \pmb{0}, I)
\end{aligned}
$$

であり，いずれも，分散共分散行列が対角行列の正規分布である．
ここで，
$p_A(\pmb{z})$ が平均 $\pmb{\mu}_A = (\mu_{A,1}, \mu_{A,2}, \ldots, \mu_{A,H})$ 分散共分散行列 $\textrm{diag}( \sigma_{A,1}^2, \sigma_{A,2}^2, \ldots, \sigma_{A,H}^2)$ の正規分布であり，
$p_B(\pmb{z})$ が平均 $\pmb{\mu}_B = (\mu_{B,1}, \mu_{B,2}, \ldots, \mu_{B,H})$ 分散共分散行列 $\textrm{diag}( \sigma_{B,1}^2, \sigma_{B,2}^2, \ldots, \sigma_{B,H}^2)$ の正規分布であるとき， $p_A(\pmb{z})$ の $p_B(\pmb{z})$ に対する KL-divergence は，次式のようになる（導出は省略）．

$$
D_{\textrm KL}(p_A(\pmb{z}) \Vert p_B(\pmb{z})) = -\frac{1}{2}\sum_{h=1}^{H} \left( 1 + \log\frac{\sigma_{A,h}^2}{\sigma_{B,h}^2} - \frac{(\mu_{A,h} - \mu_{B,h})^2}{\sigma_{B,h}^2} - \frac{\sigma_{A,h}^2}{\sigma_{B,h}^2} \right)
$$

したがって，

$$
J_2 = D_{\textrm KL}(q_{\pmb{\phi}}(\pmb{z}|\pmb{x}) \Vert p(\pmb{z}) ) = -\frac{1}{2}\sum_{h=1}^{H} (1 + \log{\sigma_h^2} - \mu_h^2 - \sigma_h^2)
$$

となる．$J_2$ の前に負号が付いているので，ELBO を最大化するためには，$J_2$ は小さければ小さいほどよい．つまり，$q_{\pmb{\phi}}(\pmb{z}|\pmb{x})$ が $p(\pmb{z})$ に近づけば近づくほど ELBO が大きくなる．


以上をまとめると，あるデータ $\pmb{x}$ に対する ELBO の値は

$$
\textrm{ELBO}(\pmb{x}; \pmb{\theta}, \pmb{\phi}) \approx -\frac{1}{2}\Vert \pmb{x} - \hat{\pmb{x}} \Vert^2 + \frac{1}{2}\sum_{h=1}^{H} (1 + \log{\sigma_h^2} - \mu_h^2 - \sigma_h^2) + \mbox{const}
$$

となる．
したがって，学習データ $\{ \pmb{x}_n \}_{n = 1}^N$ を用いて VAE を学習させるために用いる損失関数は，

$$
E(\pmb{\theta}, \pmb{\phi}) = \sum_{n=1}^N \Vert \pmb{x}_n - \hat{\pmb{x}}_n \Vert^2 - \sum_{n=1}^N\sum_{h=1}^H \left( 1 + \log{\sigma_{n,h}^2} - \mu_{n,h}^2 - \sigma_{n,h}^2 \right)
$$

となる．ニューラルネットの学習は損失関数を最小化する形で定式化するのが一般的なので，ここでは $E = -2 \sum_n \textrm{ELBO}$ として損失関数を定義している（勾配法によるパラメータの最適化においては定数項は無関係なので無視している）．


### VAE の実装上の工夫

$D$ 次元のデータを $H$ 次元に変換してから再構成する非線形AEの場合，あるデータ $\pmb{x}$ をネットワークに入力してその再構成 $\hat{\pmb{x}}$ を計算する過程は次の通りだった．

1. $\pmb{x}$ をエンコーダに入力して $H$ 次元の出力 $\pmb{z}$ を得る
1. $\pmb{z}$ をデコーダに入力して $D$ 次元の出力（再構成） $\hat{\pmb{x}}$ を得る

VAE もエンコーダとデコーダから成るが，その計算過程は少し異なり，次のようになる．

1. $\pmb{x}$ をエンコーダに入力して 2つの $H$ 次元ベクトル $\pmb{\mu}$ と $\pmb{\sigma}$ を得る
1. $N(\pmb{z}; \pmb{\mu}, \pmb{\sigma}^2I)$ に従う乱数で $H$ 次元ベクトル $\pmb{z}$ をひとつ生成する
1. $\pmb{z}$ をデコーダに入力して $D$ 次元の出力（再構成） $\hat{\pmb{x}}$ を得る

上記の 1. の計算は，非線形AEのエンコーダの出力層に2倍の数の（$2H$個の）ニューロンを置いたもので実現できる．ただし，そのままでは $\sigma_1, \sigma_2, \ldots, \sigma_H$ は正でなければならないという制約条件を入れるのが難しい．
そこで，エンコーダには $\sigma_1, \sigma_2, \ldots, \sigma_H$ のかわりに $\log{\sigma_1^2}, \log{\sigma_2^2}, \ldots, \log{\sigma_H^2}$ を出力させるのが一般的である．これらの値は任意の実数値をとる．
$\log{\sigma_h^2}$ が得られたら， $\exp\left( \frac{1}{2}\log{\sigma_h^2} \right) = \sigma_h$ として $\sigma_h$ を得ることができる．

次に 2. の計算であるが，この計算の過程にはそのままでは勾配を計算できないところがあり，PyTorch などの深層学習フレームワークでうまく実装できない．そこで，次のように計算過程を変更する：

2-1. 標準正規分布に従う乱数を$H$個生成し，$\varepsilon_1, \varepsilon_2, \ldots, \varepsilon_H$ とする

2-2. $\pmb{z}$ の要素 $z_h$ を $z_h = \mu_h + \sigma_h \varepsilon_h$ によって求める（$h = 1, 2, \ldots, H$）．

この計算によって得られる $z_h$ は平均 $\mu_h$ 分散 $\sigma_h$ の正規分布に従い，さらに勾配も計算可能となる．
この工夫を reparameterization trick という．

一方， 3. の計算は，非線形AEのデコーダと全く同じようにできる．

---
### 実験: VAEによる手書き数字画像の次元圧縮，再構成,生成

#### データを扱うクラスの定義

In [ ]:
# データを扱うためのクラス
#
class MMDataset(Dataset):

    def __init__(self, dataX):
        self.X = dataX

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X = torch.tensor(self.X[idx], dtype=torch.float32)
        return X

#### VAE の定義

In [ ]:
### VAE Encoder
#
class VAEEncoder(nn.Module):

    def __init__(self, dimX, dimHidden, dimZ):
        super().__init__()
        self.layer1        = nn.Linear(dimX, dimHidden)
        self.layer2_mu    = nn.Linear(dimHidden, dimZ)
        self.layer2_logvar = nn.Linear(dimHidden, dimZ)

    def forward(self, X):
        Y = self.layer1(X)
        Y = F.relu(Y)
        mu    = self.layer2_mu(Y)
        logvar = self.layer2_logvar(Y)
        sigma = torch.exp(0.5*logvar)
        return mu, sigma


### VAE Decoder
#
class VAEDecoder(nn.Module):

    def __init__(self, dimZ, dimHidden, dimXt):
        super().__init__()
        self.layer1 = nn.Linear(dimZ, dimHidden)
        self.layer2 = nn.Linear(dimHidden, dimXt)

    def forward(self, Z):
        Y = self.layer1(Z)
        Y = F.relu(Y)
        Xt = self.layer2(Y)
        #Xt = F.sigmoid(Xt)
        return Xt

### reparameterization trick
#
def reparameterization(mu, sigma):
    eps = torch.randn_like(sigma)
    Z = mu + sigma * eps
    return Z


### VAE
#
class VariationalAE(nn.Module):

    def __init__(self, dimX, dimHidden, dimZ):
        super().__init__()
        self.encoder = VAEEncoder(dimX, dimHidden, dimZ)
        self.decoder = VAEDecoder(dimZ, dimHidden, dimX)

    def forward(self, X):
        mu, sigma = self.encoder(X)
        Z = reparameterization(mu, sigma)
        Xt = self.decoder(Z)
        return Xt, mu, sigma

    def reconstruct(self, X):
        mu, sigma = self.encoder(X)
        Xt = self.decoder(mu)
        return Xt

    def loss(self, Xt, X, mu, sigma):
        SQE = F.mse_loss(Xt, X, reduction='sum')
        sigma2 = sigma**2
        KLD = - torch.sum(1 + torch.log(sigma2) - mu**2 - sigma2)
        return SQE, KLD

#### 学習等のための関数の定義

In [ ]:
# 学習の関数
#
def trainVAE(model, optimizer, dl):
    loss_sum = sqe_sum = kld_sum = 0.0
    n = 0
    for i, X in enumerate(dl):
        X = X.to(device)
        Xt, mu, sigma = model(X) # 一つのバッチ X を入力して出力を計算
        sqe, kld = model.loss(Xt, X, mu, sigma) # 損失関数の値を計算
        loss = sqe + kld
        optimizer.zero_grad()   # 勾配をリセット
        loss.backward()         # 誤差逆伝播でパラメータ更新量を計算
        optimizer.step()         # パラメータを更新
        n += len(X)
        loss_sum += loss.item()  # 損失関数の値
        sqe_sum += sqe.item()
        kld_sum += kld.item()

    return loss_sum/n, sqe_sum/n, kld_sum/n


# 損失関数の値を求める関数
#
@torch.no_grad()
def evaluateVAE(model, dl):
    loss_sum = sqe_sum = kld_sum = 0.0
    n = 0
    for i, X in enumerate(dl):
        X = X.to(device)
        Xt, mu, sigma = model(X)  # 一つのバッチ X を入力して出力を計算
        sqe, kld = model.loss(Xt, X, mu, sigma) # 損失関数の値を計算
        loss = sqe + kld
        n += len(X)
        loss_sum += loss.item() # 損失関数の値
        sqe_sum += sqe.item()
        kld_sum += kld.item()

    return loss_sum/n, sqe_sum/n, kld_sum/n

#### 学習

In [ ]:
# データ読み込みの仕組みを作る
dsL = MMDataset(XL2)
dsT = MMDataset(XT2)
dlL = DataLoader(dsL, batch_size=100, shuffle=True)
dlT = DataLoader(dsT, batch_size=100, shuffle=False)

In [ ]:
# ネットワークモデルの定義
H = 100
vae = VariationalAE(D, 1000, H).to(device)

# パラメータ最適化器の設定
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

# 学習の繰り返し回数
nepoch = 100

# ネットワークの構造を表示
torchsummary.summary(vae, (1, D))

# 学習
L = []
print(f'学習データ数: {len(dsL)}  テストデータ数: {len(dsT)}')
print()
print('# epoch  lossL  sqeL  kldL  lossT  sqeT  kldT')
for t in range(1, nepoch+1):
    lossL, sqeL, kldL = trainVAE(vae, optimizer, dlL)
    lossL, sqeL, kldL = lossL/D, sqeL/D, kldL/D
    lossT, sqeT, kldT = evaluateVAE(vae, dlT)
    lossT, sqeT, kldT = lossT/D, sqeT/D, kldT/D
    L.append([t, lossL, lossT])
    if (t < 10) or (t % 10 == 0):
        print(f'{t}   {lossL:.5f}  {sqeL:.5f}  {kldL:.5f}   {lossT:.5f}  {sqeT:.5f}  {kldT:.5f}')

# 学習曲線の表示
data = np.array(L)
fig, ax = plt.subplots(1, 1)
ax.plot(data[:, 0], data[:, 1], '.-', label='loss for training data')
ax.plot(data[:, 0], data[:, 2], '.-', label='loss for test data')
ax.axhline(0.0, color='gray')
ax.legend()
ax.set_title(f'loss')
plt.show()

# 学習後の損失と識別率
lossL, sqeL, kldL = evaluateVAE(vae, dlL)
print(f'# lossL: {lossL/D:.5f}', end='   ')
lossT, sqeT, kldT = evaluateVAE(vae, dlT)
print(f'# lossT: {lossT/D:.5f}')

#### 再構成

In [ ]:
# テストデータ1バッチ分の再構成
for i, X in enumerate(dlT):
    X = X.to(device)
    Xt1, mu, sigma = vae(X)
    Xt2, mu, sigma = vae(X)
    Xt3 = vae.reconstruct(X)
    break
XX     = X.to('cpu').detach().numpy() + Xm
XXrec1 = Xt1.to('cpu').detach().numpy() + Xm
XXrec2 = Xt2.to('cpu').detach().numpy() + Xm
XXrec3 = Xt3.to('cpu').detach().numpy() + Xm

# 再構成したテストデータの最初の10枚を可視化
ncol = 10
fig, ax = plt.subplots(4, ncol, figsize=(0.8*ncol, 0.8*4))

# 元画像
for j in range(ncol):
    img = XX[j, ::].reshape((28, 28))
    ax[0, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[0, j].axis('off')

# 再構成した画像（正規分布からサンプリング）
for j in range(ncol):
    img = XXrec1[j, ::].reshape((28, 28))
    ax[1, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[1, j].axis('off')

# 再構成した画像（正規分布からサンプリング）
for j in range(ncol):
    img = XXrec2[j, ::].reshape((28, 28))
    ax[2, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[2, j].axis('off')

# 再構成した画像（平均を使う）
for j in range(ncol):
    img = XXrec3[j, ::].reshape((28, 28))
    ax[3, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
    ax[3, j].axis('off')

fig.tight_layout()
plt.show()

# 再構成誤差
mse = np.mean((XX[:ncol] - XXrec3[:ncol])**2)
print(f'MSE = {mse:.5f}')

VAEの再構成は，データ $\pmb{x}$ をエンコーダに入力して正規分布のパラメータを得る → その正規分布から $\pmb{z}$ をランダムサンプリング → それをデコーダに入力して再構成 $\hat{\pmb{x}}$ を得る，という過程である．
そのため，同じ $\pmb{x}$ に対しても $\hat{\pmb{x}}$ はランダムに揺らぐ．
図の上から1行目が $\pmb{x}$ であり，2行目以降は3通りの再構成である．2行目と3行目はランダムサンプリングした2通りの $\pmb{z}$ を用いた再構成，4行目は，エンコーダが出力した平均値をそのまま $\pmb{z}$ として得られた再構成である．

#### 生成

In [ ]:
# 生成
Zgen = np.random.normal(size=50*H).reshape((50, H))
Zgen = torch.tensor(Zgen.astype(np.float32)).to(device)
XXrec = vae.decoder(Zgen)
XXrec = XXrec.cpu().detach().numpy() + Xm

# 生成した画像50枚を可視化
nrow, ncol = 5, 10
fig, ax = plt.subplots(nrow, ncol, figsize=(0.6*ncol, 0.6*nrow))
for i in range(nrow):
    for j in range(ncol):
        img = XXrec[i*ncol + j, ::].reshape((28, 28))
        ax[i, j].imshow(img, cmap=plt.cm.gray, vmin=0, vmax=1)
        ax[i, j].axis('off')

fig.tight_layout()
plt.show()